# Viteritti ViT ground-state search

Small transverse-field Ising example using the imported Viteritti-style ViT ansatz.

In [ ]:
import jax
import jax.numpy as jnp

import jVMC_exp
import jVMC_exp.nets as nets
import jVMC_exp.operator.discrete as op
import jVMC_exp.sampler as sampler
from jVMC_exp.vqs import NQS

In [7]:
Lx, Ly = 2, 2
L = Lx * Ly
g = 1.0

def site(x, y):
    return y * Lx + x

bonds = set()
for y in range(Ly):
    for x in range(Lx):
        i = site(x, y)
        bonds.add(tuple(sorted((i, site((x + 1) % Lx, y)))))
        bonds.add(tuple(sorted((i, site(x, (y + 1) % Ly)))))

hamiltonian = 0
for i, j in sorted(bonds):
    hamiltonian += -op.SigmaZ(i) * op.SigmaZ(j)
for i in range(L):
    hamiltonian += -g * op.SigmaX(i)

In [8]:
net = nets.LogViterittiSpatialViTJVMC(
    Lx=Lx,
    Ly=Ly,
    patch_size=1,
    layers=1,
    embed_dim=8,
    heads=2,
    symmetry_average="d4",
    param_dtype=jnp.float32,
    compute_dtype=jnp.float32,
)
psi = NQS(net, (Ly, Lx), batchSize=2**L, seed=1234)
exact_sampler = sampler.ExactSampler(psi)
print("Number of parameters:", psi.numParameters)

Number of parameters: 856


In [ ]:
loss_function = jVMC_exp.objective_function.Observable(hamiltonian)
stepper = jVMC_exp.stepper.Euler(2e-2)
solver = jVMC_exp.solver.Pinv(pinv_cutoff=1e-8)
opt = jVMC_exp.optimizer.MinSR(exact_sampler, psi, solver=solver, diagonalShift=1e-3)

out = opt.ground_state_search(200, loss_function, stepper)
energy = exact_sampler(hamiltonian).mean
print("Final energy:", energy)